# PySpark Basics

## Setup

In [1]:
!python --version

Python 3.10.16


In [2]:
import pyspark
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/05 08:31:58 WARN Utils: Your hostname, angela-HP-EliteBook-850-G8-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface wlp0s20f3)
25/03/05 08:31:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/05 08:31:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/05 08:31:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/05 08:31:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 36890)
Traceback (most recent call last):
  File "/home/angela/anaconda3/envs/py310/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/angela/anaconda3/envs/py310/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/home/angela/anaconda3/envs/py310/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/angela/anaconda3/envs/py310/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/home/angela/Documents/Weiterbildung/Courses/data_engineering_zoomcamp/spark-3.3.2-bin-hadoop3/python/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/home/angela/Documents/Weiterbildung/Courses/data_engineering_zoomcam

## Load some data and check the dataframe

In [6]:
df = spark.read \
    .option("header", "true") \
    .csv('../data/fhvhv_tripdata_2021-01.csv')

df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   null|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [5]:
df.schema # everything is a string

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [19]:
#!head -n 1001 ../data/fhvhv_tripdata_2021-01.csv > ../data/head.csv

In [6]:
import pandas as pd
pd.__version__

'1.5.3'

In [7]:
df_pandas = pd.read_csv("../data/head.csv")

In [8]:
df_pandas.head(5)

,hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
0,HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,NaN
1,HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,NaN
2,HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,NaN
3,HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,NaN
4,HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,NaN


In [9]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [11]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

## Spark DataFrames

### Defining the Schema

In [7]:
from pyspark.sql import types

In [8]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [9]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('../data/fhvhv_tripdata_2021-01.csv')

In [17]:
df.head(10) # schema defined

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [11]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



### Repartitioning and Writing to Files

In [19]:
df = df.repartition(24) 
df.write.parquet("../data/fhvhv/2021/01/") # when saving, the file gets repartitioned

[Stage 6:>                                                         (0 + 8) / 24]

25/03/04 22:05:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[Stage 6:===================>                                      (8 + 8) / 24]

25/03/04 22:05:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[Stage 6:==============================>                          (13 + 8) / 24]

...

[Stage 6:===================================>                     (15 + 8) / 24]

25/03/04 22:05:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[Stage 6:===================================>                     (15 + 9) / 24]

25/03/04 22:05:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


25/03/05 08:13:50 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 35726840 ms exceeds timeout 120000 ms
25/03/05 08:13:50 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/05 08:13:50 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:103)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:87)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:643)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1057)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:238)
	at 

## Spark Functions

### Default Functions

In [14]:
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID") \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

# We could do all of this in SQL -> what is the point of using Spark?

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|
|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|
|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|
|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|
|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|
|2021-01-01 00:14:30|2021-01-01 00:50:27|          71|         226|
|2021-01-01 00:22:54|2021-01-01 00:30:20|         112|         255|
|2021-01-01 00:40:12|2021-01-01 00:53:31|         255|         232|
|2021-01-01 00:56:45|2021-01-01 01:17:42|         232|         198|
|2021-01-01 00:29:04|2021-01-01 00:36:27|         113|          48|
|2021-01-01 00:48:56|2021-01-01 00:59:12|         239|          75|
|2021-01-01 00:11:53|2021-01-01 00:18:06|       

In [16]:
from pyspark.sql import functions as F # huge list of functions!

In [17]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   null| 2021-01-01|  2021-01-01|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   null| 2021-01-01|  2021-01-01|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   null| 2021-01-01|  2021-01-01|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   null| 2021-01-01|  2021-01-01|
|           HV0003| 

### User-defined Functions

In [18]:
# We have a weird business requirement:

def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    else:
        return f'e/{num:03x}'

In [19]:
crazy_stuff('B02835')

's/b13'

In [ ]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [22]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'dispatching_base_num', 'pickup_date', 'dropoff_date') \
    .show()

+-------+--------------------+-----------+------------+
|base_id|dispatching_base_num|pickup_date|dropoff_date|
+-------+--------------------+-----------+------------+
|  e/a7a|              B02682| 2021-01-01|  2021-01-01|
|  e/a7a|              B02682| 2021-01-01|  2021-01-01|
|  e/acc|              B02764| 2021-01-01|  2021-01-01|
|  e/acc|              B02764| 2021-01-01|  2021-01-01|
|  e/acc|              B02764| 2021-01-01|  2021-01-01|
|  e/9ce|              B02510| 2021-01-01|  2021-01-01|
|  e/9ce|              B02510| 2021-01-01|  2021-01-01|
|  e/acc|              B02764| 2021-01-01|  2021-01-01|
|  e/b3b|              B02875| 2021-01-01|  2021-01-01|
|  e/b3b|              B02875| 2021-01-01|  2021-01-01|
|  e/b3b|              B02875| 2021-01-01|  2021-01-01|
|  s/b13|              B02835| 2021-01-01|  2021-01-01|
|  s/b13|              B02835| 2021-01-01|  2021-01-01|
|  s/af0|              B02800| 2021-01-01|  2021-01-01|
|  s/af0|              B02800| 2021-01-01|  2021

## SQL with Spark

### Preparing the Data

In [23]:
df_green = spark.read.parquet('../data/pq/green/*/*')

In [24]:
df_green.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-12 18:15:04|  2020-01-12 18:19:52|                 N|         1|          41|          41|              1|         0.78|        5.5|  0.0|    0.

In [25]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [42]:
df_yellow = spark.read.parquet('../data/pq/yellow/*/*')

In [43]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [44]:
df_green = df_green.withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [45]:
df_yellow = df_yellow.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [46]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [47]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [33]:
common_columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [48]:
df_green_sel = df_green \
    .select(common_columns) \
    .withColumn('service_type', F.lit('green'))

In [49]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [50]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [51]:
df_trips_data.groupBy('service_type').count().show()

[Stage 15:===============================================>        (17 + 3) / 20]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [52]:
df_trips_data.registerTempTable('trips_data') # turn df into a table

/home/angela/Documents/Weiterbildung/Courses/data_engineering_zoomcamp/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


### Basic Queries

In [54]:
spark.sql("""
SELECT 
    service_type,
    count(1)
FROM 
    trips_data
GROUP BY
    service_type;
""").show()

[Stage 18:===============================================>        (17 + 3) / 20]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [59]:
df_result = spark.sql("""
    SELECT
      -- Revenue grouping
      PULocationID AS revenue_zone,
      date_trunc("month", pickup_datetime) AS revenue_month,
      service_type,
      
      -- Revenue calculation
      SUM(fare_amount) AS revenue_monthly_fare,
      SUM(extra) AS revenue_monthly_extra,
      SUM(mta_tax) AS revenue_monthly_mta_tax,
      SUM(tip_amount) AS revenue_monthly_tip_amount,
      SUM(tolls_amount) AS revenue_monthly_tolls_amount,
      SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
      SUM(total_amount) AS revenue_monthly_total_amount,
      
      -- Additional calculations
      COUNT(*) AS total_monthly_trips,
      AVG(passenger_count) AS avg_monthly_passenger_count,
      AVG(trip_distance) AS avg_monthly_trip_distance
    FROM
      trips_data
    GROUP BY
      1,
      2,
      3
    """)

In [66]:
df_result.show(5)

[Stage 36:=====================================================>  (19 + 1) / 20]

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+-------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|total_monthly_trips|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+-------------------+---------------------------+-------------------------+
|         127|2020-02-01 00:00:00|       green|  21571.680000000

In [65]:
# Write the results to our 'data lake'
df_result.write.parquet("../data/report/revenue/")

### Group By

In [69]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [70]:
df_green.registerTempTable('green')

In [88]:
df_green_revenue = spark.sql("""
    SELECT 
        PULocationID AS zone, 
        date_trunc("hour", pickup_datetime) AS hour, 
        SUM(total_amount) AS amount,
        COUNT(*) AS record_count
    FROM 
        green
    WHERE
        pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY
        zone,
        hour
        ;
""")

In [96]:
df_green_revenue \
    .repartition(20) \
    .write.parquet("../data/report/revenue/green", mode="overwrite")

[Stage 89:==================================================>       (7 + 1) / 8]

25/03/05 11:30:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/03/05 11:30:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [91]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [92]:
df_yellow.registerTempTable('yellow')

In [93]:
df_yellow_revenue = spark.sql("""
    SELECT 
        PULocationID AS zone, 
        date_trunc("hour", pickup_datetime) AS hour, 
        SUM(total_amount) AS amount,
        COUNT(*) AS record_count
    FROM 
        yellow
    WHERE
        pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY
        zone,
        hour
        ;
""")

In [97]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet("../data/report/revenue/yellow", mode="overwrite")

[Stage 97:>                                                         (0 + 8) / 9]

25/03/05 11:31:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/03/05 11:31:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


### Loading previously materialized results

In [111]:
df_yellow_revenue = spark.read.parquet("../data/report/revenue/yellow")
df_green_revenue = spark.read.parquet("../data/report/revenue/green")

In [114]:
df_green_revenue.printSchema()

root
 |-- zone: integer (nullable = true)
 |-- hour: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- record_count: long (nullable = true)



### Joins in Spark

In [115]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('record_count', 'green_record_count') 
df_yellow_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('record_count', 'yellow_record_count') 

In [116]:
# Two large tables
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [118]:
df_join.show(5)

+-------------------+----+------------------+------------------+------------------+-------------------+
|               hour|zone|      green_amount|green_record_count|     yellow_amount|yellow_record_count|
+-------------------+----+------------------+------------------+------------------+-------------------+
|2020-01-01 00:00:00|  22|              15.8|                 1|              15.8|                  1|
|2020-01-01 00:00:00|  25|             531.0|                26|             531.0|                 26|
|2020-01-01 00:00:00|  55|129.29000000000002|                 4|129.29000000000002|                  4|
|2020-01-01 00:00:00|  56|             99.69|                 3|             99.69|                  3|
|2020-01-01 00:00:00|  60|            160.04|                 6|            160.04|                  6|
+-------------------+----+------------------+------------------+------------------+-------------------+
only showing top 5 rows



In [117]:
df_join.write.parquet("../data/report/revenue/total", mode='overwrite')

25/03/05 12:01:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [121]:
# One large and one small table
df_zones = spark.read.csv("../data/taxi_zone_lookup.csv", header=True)
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [122]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [126]:
df_result.drop('zone').write.parquet('../data/tmp/revenue-zones')

[Stage 142:>                                                        (0 + 8) / 8]

25/03/05 12:10:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
